# Assignment Text summariser powered by LLM

In [1]:
# this notebook is suitable to run on colab
# If using vscode, execute requirements.txt in fresh enviroment, python=3.10

#from google.colab import drive
#drive.mount('/content/drive')

In [44]:
# Import necessary libraries
import numpy as np
import pandas as pd

import torch
import transformers
import evaluate
#!pip install evaluate
from tqdm import tqdm
from transformers.optimization import Adafactor
from torch.utils.data import DataLoader
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from bert_score import score

### Checking prerequisites

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(0))
print("PyTorch version:", torch.__version__)

In [ ]:
# Switch device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [6]:
# Checking version is very important when working on different compilers
print("Torch version:", torch.__version__)
print("Torch location:", torch.__file__)
print("Transformers version:", transformers.__version__)

Torch version: 2.11.0+cpu
Torch location: c:\Users\Sandeep\anaconda3\envs\summ\lib\site-packages\torch\__init__.py
Transformers version: 5.13.1


## 1.0 Loading Dataset

In [ ]:
# VS code version
dataset = load_from_disk("S:\Projects\Datasets\TextS\samsum_dataset")

# Colab version
#dataset = load_from_disk("/content/drive/MyDrive/samsum_dataset")

##### Neural networks cannot process raw text directly. Text must first be converted into numerical representations. Modern LLMs achieve this using subword tokenization. LLM's like gpt rely on tokenization methods like BPE, but for this dataset and T5 llm, SentencePiece is used. Lets take a look at token embedding and trannsformer, Text to text transfer transformer, Flan-T5-base (trained on 250M parameters), big enough to be called LLM

### 1.1 Load LLM (FLAN-t5)

In [8]:
# Loading the flan-t5 in vanilla form
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1740.11it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### 1.2 Lets inspect the model

In [10]:
model.config

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": false,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {


In [12]:
# Model embedding
model.shared

Embedding(32128, 768)

In [ ]:
model.encoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerFF(
          (DenseReluDense): T5DenseGatedActDense(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
            (wo): Linear(in_features=2048, out_features=768, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
      

In [ ]:
model.decoder

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerCrossAttention(
          (EncDecAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=F

In [ ]:
model.lm_head

Linear(in_features=768, out_features=32128, bias=False)

#### The main difference in encoder and decoder as seen above, encoder dosent have cross attention layer. The encoder only reads the input sentence, and dosent need another sequence to attend to.

#### The decoder has already generated summary, but it also takes input from enncoder along with generated input which is cross verified while generating the optimal output. This process is also called self supervising learning.

### 1.3 Lets look at tokenizer

In [62]:
print(tokenizer.vocab_size)
print(tokenizer.model_max_length)
print(tokenizer.pad_token)
print(tokenizer.eos_token)

32100
512
<pad>
</s>


In [16]:
# Lets take look at sample and tokenize them and look at their token ids
sample = dataset["train"][0]["dialogue"]
print(sample)

tokens = tokenizer.tokenize(sample)
print("\n",tokens[:50])

ids = tokenizer.encode(sample)
print("\n",ids[:50])

Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

 ['▁Amanda', ':', '▁I', '▁baked', '▁cookies', '.', '▁Do', '▁you', '▁want', '▁some', '?', '▁Jerry', ':', '▁Sure', '!', '▁Amanda', ':', '▁I', "'", 'll', '▁bring', '▁you', '▁tomorrow', '▁', ':', '-', ')']

 [21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128, 58, 16637, 10, 10625, 55, 21542, 10, 27, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1]


### 1.4 Model already has pretrtained embedding lets extract one

In [17]:
embeddings = model.get_input_embeddings().weight
print(embeddings.shape)

torch.Size([32128, 768])


In [18]:
tokens = tokenizer.tokenize("hello")
print(tokens)

token = tokens[0]

token_id = tokenizer.convert_tokens_to_ids(token)

print(token)
print(token_id)

['▁hello']
▁hello
21820


##### The pretrainied embedding has 21820 as hello, i will make more sense when compared with surounding vectors and it will have closer semantic meaning. T5 base dont need a positional embedding, its has Relative position, biased inside the attention mechanism

### 1.5 Now lets take a look at attention block

In [ ]:
print(model.encoder.block[0].layer[0])

T5LayerSelfAttention(
  (SelfAttention): T5Attention(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)
    (o): Linear(in_features=768, out_features=768, bias=False)
    (relative_attention_bias): Embedding(32, 12)
  )
  (layer_norm): T5LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
)


In [ ]:
attn = model.encoder.block[0].layer[0].SelfAttention
print(attn)

T5Attention(
  (q): Linear(in_features=768, out_features=768, bias=False)
  (k): Linear(in_features=768, out_features=768, bias=False)
  (v): Linear(in_features=768, out_features=768, bias=False)
  (o): Linear(in_features=768, out_features=768, bias=False)
  (relative_attention_bias): Embedding(32, 12)
)


In [ ]:
print(attn.has_relative_attention_bias)

True


##### This shows that the model has relative attention type, rather than absolute that we see in classic gpt

In [ ]:
print(attn.relative_attention_bias)

Embedding(32, 12)


##### Here 32 is the relative distance buckets with 12 attention heads, model learns distance rather than positions, which is a smarter approach

In [ ]:
# Attention weights
attn.relative_attention_bias.weight

Parameter containing:
tensor([[ 3.3072e+00, -1.4124e+01,  2.2363e+00, -7.5515e+00,  8.4037e+00,
          5.4025e+00,  4.9113e-01,  2.5243e-01,  4.3401e+00,  6.6022e+00,
         -8.6801e+00, -2.5473e+01],
        [-2.5756e+01,  1.0481e+01,  8.4726e+00,  3.9471e+00,  9.8540e+00,
          1.7485e+00,  9.1644e+00,  6.1179e+00,  7.9472e+00, -4.2284e+00,
          2.8060e+00,  7.6758e+00],
        [-1.5956e+01,  8.7715e+00,  5.2965e+00,  4.5750e+00,  7.7746e+00,
          9.5001e-01,  8.6429e+00,  6.6384e+00,  7.5241e+00, -1.7510e+01,
          3.7001e+00,  8.0501e+00],
        [-1.5508e+01,  7.6623e+00,  4.6198e+00,  4.7793e+00,  6.7673e+00,
          1.9559e+00,  8.1014e+00,  6.7059e+00,  7.0760e+00, -1.9015e+01,
          3.9715e+00,  8.0325e+00],
        [-1.3945e+01,  7.0022e+00,  4.4271e+00,  4.7923e+00,  5.8716e+00,
          2.2086e+00,  7.6472e+00,  6.8344e+00,  6.7654e+00, -2.1642e+01,
          4.1278e+00,  7.9277e+00],
        [-1.5966e+01,  6.4181e+00,  4.3777e+00,  4.9517e+0

##### These are pretrained weights

## 2.0 PreProcesssing

In [ ]:
# Dataset in use. Dataset paths is loaded above
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [ ]:
# Look at a sample and add summary
sample = dataset["train"][0]

prompt = f"""Summarize the following conversation.

Dialogue:
{sample['dialogue']}

Summary:"""

print(prompt)

Summarize the following conversation.

Dialogue:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

Summary:


### 2.1 Applying the above example to whole dataset, because for FLAN an instruction prompt will give out better results

In [22]:
# always load fresh
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

MAX_INPUT_LENGTH = 384
MAX_TARGET_LENGTH = 64

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3361.98it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [23]:
# Helper function to convert the dialouge to an instruction based prompt
def create_prompt(dialogue):
    return f"""Summarize the following conversation.

Dialogue:
{dialogue}

Summary:"""

In [24]:
# helper function to tokenize dialouge and traget summaries
def preprocess_function(batch):

    prompts = [
        create_prompt(dialogue)
        for dialogue in batch["dialogue"]
    ]

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [25]:
# helper function, applies the above changes to the entire dataset
def transform(dataset):        
        tokenized_dataset = dataset.map(
            preprocess_function,
            batched=True,
            remove_columns=dataset["train"].column_names,
            desc="Tokenizing dataset",
        )

        return tokenized_dataset

In [26]:
# Apply changes
tokenized_dataset = transform(dataset)

Tokenizing dataset: 100%|██████████| 818/818 [00:00<00:00, 5917.82 examples/s]


In [29]:
# Check details from the sample
sample = tokenized_dataset["train"][0]

print(sample.keys())

print(sample["input_ids"][:20])
print(sample["labels"][:20])

print(tokenizer.decode(sample["input_ids"]))
print(tokenizer.decode(
    [x for x in sample["labels"] if x != -100]
))

dict_keys(['input_ids', 'attention_mask', 'labels'])
[12198, 1635, 1737, 8, 826, 3634, 5, 5267, 10384, 10, 21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128]
[21542, 13635, 5081, 11, 56, 830, 16637, 128, 5721, 5, 1]
Summarize the following conversation. Dialogue: Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-) Summary:</s>
Amanda baked cookies and will bring Jerry some tomorrow.</s>


## 3.0 Fine tuning the data and model

In [30]:
# Helpler function to creates data collator

def create_data_collator(tokenizer, model):
    
    return DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        return_tensors="pt"
    )

In [32]:
# Helper function to create data loader

def create_dataloader(tokenized_dataset, data_collator, batch_size=2):
    
    train_loader = DataLoader(
        tokenized_dataset["train"],
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator,
        pin_memory=True,
        num_workers=2
    )

    return train_loader

#### The collator creates the decoder_input_ids automatically,, this process is called "teacher forcing". At every step, the decoder is shown the "correct" previous word to verify the result, not its own prediction (that can be wrong).

In [37]:
# Helper funcction used during training process
def create_optimizer(model):
    """
    Create the Adafactor optimizer.
    """

    optimizer = Adafactor(
        model.parameters(),
        lr=None,
        scale_parameter=True,
        relative_step=True,
        warmup_init=True
    )

    return optimizer

In [33]:
# Important helper function, generates a sample summary after every epoch
def evaluate_sample(
    model,
    tokenizer,
    prompt,
    device,
    max_input_length=384,
    max_new_tokens=64
):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    )

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    model.train()

    return summary

In [34]:
# Helper to save checkpoints, use full in colab when runtime disconnects
def save_checkpoint(
    model,
    tokenizer,
    save_directory,
    epoch
):
    """
    Save the model and tokenizer after each epoch.
    """

    checkpoint_path = f"{save_directory}/epoch_{epoch}"

    model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)

    print(f"Saved checkpoint: {checkpoint_path}")

In [ ]:
# Heper funciton to train the model
def train_model(
    model,
    tokenizer,
    train_loader,
    optimizer,
    dataset,
    device,
    epochs=3,
    save_directory="/content/drive/MyDrive/flan_t5_samsumV2"
):
    """
    Fine-tune FLAN-T5 on the SAMSum dataset.
    """

    model.to(device)
    model.train()

    optimizer.zero_grad(set_to_none=True)

    fixed_prompt = create_prompt(
        dataset["validation"][0]["dialogue"]
    )

    for epoch in range(epochs):

        total_loss = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{epochs}"
        )

        for batch in progress_bar:

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = model(**batch)

            loss = outputs.loss

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            optimizer.step()

            optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        avg_loss = total_loss / len(train_loader)

        print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")

        sample_summary = evaluate_sample(
            model,
            tokenizer,
            fixed_prompt,
            device
        )

        print(f"Sample Summary:\n{sample_summary}\n")

        save_checkpoint(
            model,
            tokenizer,
            save_directory,
            epoch + 1
        )

In [38]:
# Trainig function ### ONLY TRAIN WHEN MODEL IS NOT ATTACHED, OR IF GPU IS PRESENT Get model at: (https://huggingface.co/San0160/Text_summariser_FLAN_T5)
# Wrap this in exception handler in src code, add if else, add logger.
data_collator = create_data_collator(tokenizer, model)

train_loader = create_dataloader(tokenized_dataset, data_collator)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

optimizer = create_optimizer(model)

'''
train_model(
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    optimizer=optimizer,
    dataset=dataset,
    device=device,
    epochs=3
)
'''

'\ntrain_model(\n    model=model,\n    tokenizer=tokenizer,\n    train_loader=train_loader,\n    optimizer=optimizer,\n    dataset=dataset,\n    device=device,\n    epochs=3\n)\n'

## 4.0 Testing the model

In [39]:
# Load model
model_path = r"S:\Projects\Datasets\TextS\model\working_model" 

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

print(f"Model and tokenizer loaded successfully from {model_path}")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1120.07it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model and tokenizer loaded successfully from S:\Projects\Datasets\TextS\model\working_model


In [ ]:
# Function reused, it has to be same function used during training for correct results and evaluation
def create_prompt(dialogue):
    return f"""Summarize the following conversation.

Dialogue:
{dialogue}

Summary:"""

In [ ]:
# Helper to apply model to the uploaded dialouge
def summarize(dialogue):
    prompt = create_prompt(dialogue)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=384
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary

In [42]:
# Test on various samples
dialogue = """
Liam: I still can’t believe what happened in last night’s game.
Noah: Mate, I’m still recovering. That final ten minutes were chaos.
Liam: The equaliser alone felt like a movie scene.
Noah: And the crowd went absolutely wild, I thought the stadium would explode.
Liam: Did you see the coach’s face? Pure disbelief.
Noah: He wasn’t expecting the substitute to pull that off.
Liam: Honestly, neither was I. That volley was ridiculous.
Noah: The keeper didn’t even move, he just watched it fly in.
Liam: That’s when I knew the momentum had shifted.
Noah: Same. You could feel the energy change instantly.
Liam: But the real shock was the winning goal.
Noah: Oh absolutely, that counterattack was textbook.
Liam: Three passes, one sprint, and boom—top corner.
Noah: I swear I jumped so high I nearly spilled my drink.
Liam: I did spill mine. Worth it though.
Noah: The defenders looked exhausted by then.
Liam: They’d been pressing nonstop for eighty minutes.
Noah: And it finally caught up with them.
Liam: The captain’s reaction after scoring was priceless.
Noah: He just collapsed on the pitch, arms wide open.
Liam: You could tell how much it meant to him.
Noah: And to the fans too. That roar was unreal.
Liam: I still have a headache from it.
Noah: Same, but I’m not complaining.
Liam: So, do you think this win changes their season?
Noah: Definitely. It’s the kind of victory that shifts confidence.
Liam: I agree. They needed a spark, and they got one.
Noah: Last night wasn’t just a win—it was a statement.
Liam: A loud one.
Noah: And now everyone’s watching.
Liam: Let’s see if they can keep it going.
Noah: I think they will.
Liam: After last night, I finally believe it.

"""

print("Generated Summary:")
print(summarize(dialogue))

Generated Summary:
Liam and Noah can't believe what happened in last night's game. The final ten minutes were chaotic. The coach's face was disbelieving. The winning goal was textbook. The captain collapsed on the pitch.


## 5.0 Evaluation Metrics

In [ ]:
# Get predictions for entire test dataset
predictions = []
references = []
dialogues = []

for sample in tqdm(dataset["test"]):
    dialogue = sample["dialogue"]
    reference = sample["summary"]

    prediction = summarize(dialogue)

    dialogues.append(dialogue)
    references.append(reference)
    predictions.append(prediction)

results_df = pd.DataFrame({
    "dialogue": dialogues,
    "reference": references,
    "prediction": predictions
})

results_df.head()

In [ ]:
# Save the results
results_df.to_csv("flan_t5_predictions.csv", index=False)
print("Predictions saved successfully!")

Predictions saved successfully!


In [ ]:
# Load back if needed.
results_df = pd.read_csv("flan_t5_predictions.csv")

### 5.1 ROUGE

In [56]:
# Load ROUGE
rouge = evaluate.load("rouge")

rouge_results = rouge.compute(
    predictions=results_df["prediction"].tolist(),
    references=results_df["reference"].tolist(),
    use_stemmer=True
)

print(rouge_results)

{'rouge1': np.float64(0.5199562497425869), 'rouge2': np.float64(0.27048264066628636), 'rougeL': np.float64(0.4326341229906171), 'rougeLsum': np.float64(0.433031409207956)}


In [60]:
# Print ROUGE cleaner version (resuts discussion in report)
rouge_report = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Score (%)": [
        rouge_results["rouge1"] * 100,
        rouge_results["rouge2"] * 100,
        rouge_results["rougeL"] * 100,
    ]
})

print(rouge_report)

# Save rouge report (display on site)
rouge_report.to_csv("rouge_scores.csv", index=False)
print("\n ROUGE scores saved successfully!")

    Metric  Score (%)
0  ROUGE-1  51.995625
1  ROUGE-2  27.048264
2  ROUGE-L  43.263412

 ROUGE scores saved successfully!


### 5.2 BERT

In [ ]:
# BERTscore evaluation
P, R, F1 = score(
    cands=results_df["prediction"].tolist(),
    refs=results_df["reference"].tolist(),
    lang="en",
    verbose=True
)

c:\Users\Sandeep\anaconda3\envs\summ\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sandeep\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3583.66it/s]
[transformers] RobertaModel LOAD REPORT from: ro

calculating scores...
computing bert embedding.


100%|██████████| 26/26 [05:35<00:00, 12.92s/it]


computing greedy matching.


100%|██████████| 13/13 [00:00<00:00, 26.31it/s]


done in 336.38 seconds, 2.43 sentences/sec


In [41]:
bert_results = {
    "Precision": P.mean().item(),
    "Recall": R.mean().item(),
    "F1": F1.mean().item()
}

print(bert_results)

{'Precision': 0.9269626140594482, 'Recall': 0.9222183227539062, 'F1': 0.9244028329849243}


In [ ]:
# Cleaner results, Discussion in report
bert_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1"],
    "Score": [
        bert_results["Precision"] * 100,
        bert_results["Recall"] * 100,
        bert_results["F1"] * 100,
    ]
})

bert_df

,Metric,Score
0,Precision,92.696261
1,Recall,92.221832
2,F1,92.440283


In [ ]:
# Save
bert_df.to_csv("bert_scores.csv", index=False)
print("BERTscores saved successfully!")